In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [3]:
# connect to Chroma, use Huggingface embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# setup the langchain objects: retriever and llm
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [5]:
# using the langchain invoke method.
retriever.invoke('Where are HomeHaven stores located?')

[Document(id='79bf1d6c-f4dd-4ec2-ae92-d06e2ce60b03', metadata={'doc_type': 'Reference_Tech', 'source': '..\\KnowledgeBase\\Reference_Tech\\HH-65_STORE_SERVICES.md'}, page_content='---\ndoc_id: HH-65\ntitle: In-Store Services Overview\ndoc_type: reference\nversion: 1.0\nlast_updated: 2026-01-12\n---\n\n# In-Store Services Overview (HH-65)\n\n## 1. Overview\nWhile **HomeHaven** is a digital-first retailer, our physical Experience Centers across the UAE serve as critical hubs for logistics, sustainability, and face-to-face customer support. Each location is equipped to handle omnichannel fulfillment and post-purchase service.\n\n## 2. Core In-Store Services\n\n### Click & Collect (Ready in 2 Hours)\n*   **Availability:** Offered at all three flagship locations.\n*   **Process:** Select "Store Pickup" at checkout. You will receive an SMS/Email with a unique QR code once your items are staged and ready.\n*   **Holding Period:** Orders are held for **3 days**. After this period, the items ar

In [6]:
llm.invoke("Where are HomeHaven stores located?")

AIMessage(content='HomeHaven stores are primarily located in the United States. They can be found in various states, often within shopping centers or retail districts. For the most accurate and up-to-date information on specific store locations, I recommend visiting the HomeHaven website or contacting their customer service directly.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 15, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_f276620e06', 'id': 'chatcmpl-DH5TtYyZI2JefWy4Ja1jLsRgrwtIo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cccf3-35ed-7c42-b7a3-3034ec22eee3-0', tool_calls=[], invalid_tool_calls=[], 

In [7]:
## putting it all together
SYSTEM_PROMPT_TEMPLATE = """
You are an expert question answerer for HomeHaven, a retailer of home goods.
You have access to a retriever that can retrieve relevant information about HomeHaven from a vector database.
Use the retriever to find relevant information and answer the user's question as accurately as possible.
If you don't know the answer, say you don't know. Do not make up an answer.
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Where are HomeHaven stores located?", [])

'HomeHaven has physical Experience Centers across the UAE. These locations serve as hubs for logistics, sustainability, and face-to-face customer support. However, the specific cities or addresses of these stores are not detailed in the provided documents.'

In [10]:
# create a gradio interface
gr.ChatInterface(answer_question).launch()

c:\Users\mom_e\Projects\RAG_for_Retailer\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
